In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, PoissonRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score,
)

In [34]:
df = pd.read_csv("./dataset/education_career_success_cleaned.csv")
df.head()

,Student_ID,Age,Gender,High_School_GPA,SAT_Score,University_GPA,Field_of_Study,Internships_Completed,Projects_Completed,Certifications,Soft_Skills_Score,Networking_Score,Job_Offers,Starting_Salary,Career_Satisfaction,Years_to_Promotion,Current_Job_Level,Work_Life_Balance,Entrepreneurship
0,S001,22,Male,3.8,1450,3.6,Computer Science,3,7,2,8,7,3,85000,8,2,Mid,7,No
1,S002,24,Female,3.6,1380,3.4,Business,2,5,3,7,6,2,65000,7,3,Mid,6,No
2,S003,21,Male,3.9,1520,3.8,Engineering,4,9,4,9,8,4,120000,9,1,Senior,6,No
3,S004,23,Female,3.5,1300,3.2,Psychology,1,3,1,6,5,1,48000,6,4,Entry,8,No
4,S005,25,Male,3.7,1420,3.5,Medicine,2,6,2,8,7,3,95000,8,2,Mid,5,No


In [35]:
df2 = df.copy()

In [36]:
df2.drop(['High_School_GPA', 'SAT_Score', 'Career_Satisfaction', 'Years_to_Promotion','Current_Job_Level', 'Work_Life_Balance', 'Entrepreneurship', 'Student_ID'], axis=1, inplace=True)
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    400 non-null    int64  
 1   Gender                 400 non-null    object 
 2   University_GPA         400 non-null    float64
 3   Field_of_Study         400 non-null    object 
 4   Internships_Completed  400 non-null    int64  
 5   Projects_Completed     400 non-null    int64  
 6   Certifications         400 non-null    int64  
 7   Soft_Skills_Score      400 non-null    int64  
 8   Networking_Score       400 non-null    int64  
 9   Job_Offers             400 non-null    int64  
 10  Starting_Salary        400 non-null    int64  
dtypes: float64(1), int64(8), object(2)
memory usage: 34.5+ KB


In [37]:
best_model_name = None
best_model_pipeline = None
best_r2_score = -float("inf")

In [38]:
target = "Job_Offers"
X = df2.drop(columns=[target])
y = df2[target]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Poisson Regression": PoissonRegressor(max_iter=1000),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    preds_rounded = np.clip(np.round(preds), 0, None).astype(int)
    y_test_int = y_test.astype(int)
    acc = accuracy_score(y_test_int, preds_rounded)
    f1 = f1_score(y_test_int, preds_rounded, average="weighted", zero_division=0)

    results.append({
        "Model": name,
        "MAE": round(mae, 3),
        "MSE": round(mse, 3),
        "RMSE": round(rmse, 3),
        "R2": round(r2, 3),
        "Accuracy": round(acc, 3),
        "F1_Score": round(f1, 3),
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
print(results_df.to_string(index=False))

             Model   MAE   MSE  RMSE    R2  Accuracy  F1_Score
     Random Forest 0.012 0.002 0.046 0.999     1.000     1.000
 Gradient Boosting 0.007 0.002 0.046 0.999     1.000     1.000
  Ridge Regression 0.142 0.033 0.180 0.983     0.988     0.987
 Linear Regression 0.143 0.033 0.181 0.983     0.988     0.987
Poisson Regression 0.248 0.092 0.303 0.953     0.950     0.938


In [39]:
if r2 > best_r2_score:
        best_r2_score = r2
        best_model_name = name

In [40]:
import joblib

# Susun bundle dictionary
bundle = {
    "model": best_model_pipeline,  # Objek model/pipeline yang sudah di-fit
    "categories": {
        "Gender": ["Male", "Female", "Other"],
        "Field_of_Study": [
            "Computer Science",
            "Engineering",
            "Business",
            "Medicine",
            "Arts",
            "Mathematics",
            "Law",
        ],
    },
    "features": [
        "Age",
        "Gender",
        "University_GPA",
        "Field_of_Study",
        "Internships_Completed",
        "Projects_Completed",
        "Certifications",
        "Soft_Skills_Score",
        "Networking_Score",
        "Starting_Salary",
    ],
}

# Simpan ke .pkl
joblib.dump(bundle, "model_career_prediction.pkl")

['model_career_prediction.pkl']